# 02 · Fire history and its limits

Public-source research and teaching prototype, version 0.1.0. Prepared by Lilly Jones, PhD. No proposed partnership is assumed to be confirmed.


In [ ]:
from pathlib import Path
import sys
candidate = Path.cwd().resolve()
ROOT = next(p for p in [candidate, *candidate.parents] if (p / "project.yml").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import yaml
from IPython.display import display, Markdown, HTML


## Select a regional context window
Choose one of the four context windows. Its records are not assigned to a Nation, and its box is not a territorial boundary. This notebook reads the bundled real MTBS subset; it makes no network request to obtain observations.

In [ ]:
from esiilfire.io import history_manifest
from esiilfire.present import history_map, history_chart
import geopandas as gpd
manifest = history_manifest()
REGION = "northern-plains"  # southwest-colorado, central-arizona, klamath-region
context = manifest["regions"][REGION]
display(Markdown(context["note"]))
display({"source":manifest["source"], "start_year":manifest["start_year"], "end_year":manifest["end_year"], "original_download_date":manifest["original_download_date"]})

## Reconcile counts before interpreting them
One event ID is counted once per window. “Prescribed Fire” remains a recorded source category, not a cultural-fire classification. No mapped event is different from no fire.

In [ ]:
annual = pd.read_csv(ROOT / "data/processed" / f"history_{REGION}.csv")
events = gpd.read_file(ROOT / "data/processed" / f"history_{REGION}.geojson")
assert annual.mapped_events.sum() == len(events) == context["mapped_events"]
if len(events):
    assert not events.event_id.duplicated().any()
display(annual.groupby("type", sort=False)[["mapped_events", "acres_in_window"]].sum())

In [ ]:
display(HTML(history_map(REGION)))
display(HTML(history_chart(REGION)))

## Investigate limitations
The public snapshot ends in 2024. MTBS generally maps large fires using 1,000-acre western and 500-acre eastern thresholds; small stewardship burns can be missed. The event-area sum counts reburns more than once and is not unique land area burned. Display polygons are simplified, while regional acreage is calculated from unsimplified intersections.

Exercise: identify one pattern, two explanations the archive cannot distinguish, and the additional evidence needed. Explain why a lack of recorded prescribed fires cannot establish a lack of cultural burning. [MTBS methods](https://www.mtbs.gov/faq/fire-mapping).